# PPML 1-D TM — quickstart

Run the Python port and compare to the committed MATLAB golden — no MATLAB/Octave needed. Executes top-to-bottom in CI.

In [ ]:
import sys
from pathlib import Path
import numpy as np
from scipy.io import loadmat
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'python_src'))
from ppml_1d_tm import RTA_1d_tm
print('port imported')

## One fixture vs MATLAB golden

In [ ]:
fid = 'scenario_S1_OpticalCritical_nu3.0_th30'
fx = loadmat(ROOT/'fixtures'/f'{fid}.mat', struct_as_record=False, squeeze_me=True)['fx']
ref = loadmat(ROOT/'reference_outputs'/'matlab'/f'{fid}.mat', struct_as_record=False, squeeze_me=True)['ref']
RR, TT, AA = RTA_1d_tm(fx.a, int(fx.L), complex(fx.epssup), complex(fx.epssub),
                       fx.epsxA, fx.epszA, fx.epsxB, fx.epszB, fx.sigma, fx.f, fx.d,
                       int(fx.halfnpw), float(fx.k0), float(fx.kpar))
print(f'Python : R={RR.real:.6f} T={TT.real:.6f}')
print(f'MATLAB : R={complex(ref.RR).real:.6f} T={complex(ref.TT).real:.6f}')
print(f'max|py-matlab| = {max(abs(RR-complex(ref.RR)), abs(TT-complex(ref.TT))):.2e}')
assert abs(RR-complex(ref.RR)) < 1e-9

## All physical fixtures on the y = x line

In [ ]:
sys.path.insert(0, str(ROOT / 'verify'))
from plots import collect
ref_re, py_re, ref_im, py_im = collect(ROOT, 'matlab')
x = np.concatenate([ref_re, ref_im]); y = np.concatenate([py_re, py_im])
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(x, y, s=4, alpha=0.3)
lim = max(1.0, float(np.nanmax(np.abs(x))))
ax.plot([-lim,lim],[-lim,lim],'r',lw=1)
ax.set_xlabel('MATLAB reference'); ax.set_ylabel('Python port'); ax.set_aspect('equal')
ax.set_title(f'{x.size} values on y=x'); plt.show()
print(f'{x.size} values compared')